# Chapter 3

## Summarizing a document bigger than the LLM’s context window

In [3]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [4]:
from langchain_anthropic import ChatAnthropic
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import getpass

In [5]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [6]:
llm = ChatAnthropic(api_key=OPENAI_API_KEY,model_name="claude-haiku-4-5-20251001")

In [7]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            for text_chunk in 
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [8]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()        
        }
    )
)

In [9]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

In [10]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)     

In [11]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [12]:
print(summary)

# Consolidated Summary of Moby-Dick, Chapters 1-7

Ishmael, the narrator, seeks passage on a whaling voyage as a remedy for depression and existential despair. Drawn by the sea's mystical appeal and the promise of adventure, he chooses to work as a sailor rather than travel as a passenger.

After arriving in New Bedford in December, Ishmael finds lodging at the Spouter-Inn, a whaling establishment decorated with maritime artifacts and a mysterious painting. Unable to secure a private room, he reluctantly agrees to share a bed with a harpooner named Queequeg, a heavily tattooed Polynesian from the South Seas.

Initially terrified by Queequeg's exotic appearance, scarred features, and strange possessions—including shrunken heads and a wooden idol—Ishmael's fear stems from cultural prejudice. When Queequeg performs a ritualistic prayer and climbs into bed brandishing a tomahawk, Ishmael panics. However, the landlord assures him of his safety, and once Queequeg agrees to cease smoking, Ish

## Summarizing across documents

In [13]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [14]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [15]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [16]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import PromptTemplate
import getpass

In [17]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [18]:
llm = ChatAnthropic(api_key=OPENAI_API_KEY,model_name="claude-haiku-4-5-20251001")

In [19]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [20]:
refine_summary_template = """
You must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [21]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)
        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [22]:
full_summary = refine_summary(all_docs)
print(full_summary)

{'final_summary': "# Final Summary\n\n**Paestum: An Ancient Greek City in Magna Graecia**\n\nPaestum is a frazione (village) of the comune (municipality) of Capaccio Paestum, located in the Cilento area of the province of Salerno in the Campania region of southern Italy. It lies on the Tyrrhenian coast approximately 22 miles (35 km) southeast of modern Salerno and is notable for the famous ruins of the ancient city of the same name nearby.\n\nOriginally named Poseidonia by Greek settlers from Sybaris around 600 BCE, the ancient city thrived as a Greek settlement for approximately two centuries before experiencing a series of conquests and transformations. Ancient sources attribute Sybaris's founding to the Achaeans, with Is of Helice named as its founder. During its prosperous Greek period, Poseidonia enjoyed the status of an autonomous Greek polis and was endowed with a defensive wall featuring four gates, probably built in phases. The Lucanians, an Oscan-speaking Italic people with e